<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# E-Commerce Analytics 360
## Étape 1 — Contexte métier & exploration des données
### ✅ VERSION CORRIGÉE

> **Comment lire ce corrigé :**  
> Chaque étape est précédée d'un bloc qui explique le **POURQUOI** avant le **COMMENT**.  
> Les blocs `MÉTHODE` expliquent les choix techniques et les bonnes pratiques.  
> Les blocs `INTERPRÉTATION` lisent les résultats comme le ferait un analyste en restitution.  
> Les blocs `MÉTIER` font le lien entre le chiffre et la décision business.

| | |
|---|---|
| **Niveau** | Intermédiaire |
| **Outils** | Python — pandas, matplotlib |
| **Durée estimée** | 1h à 2h |

---
## 0. Réflexion stratégique

> **MÉTHODE — Réfléchir avant de coder : l'étape que les analystes juniors sautent.**
>
> Cette section force la réflexion **métier** avant la réflexion **technique**. Les réponses ici ne sont pas définitives — elles sont des hypothèses de travail qu'on testera avec les données.

### Question 1 — Formule de la marge

```
Marge brute (€)  = prix_unitaire - cout_produit
Taux de marge (%) = (prix_unitaire - cout_produit) / prix_unitaire × 100
```

**Exemple :** un produit vendu 50€ acheté 30€ → marge de 20€ (40%).

> **MÉTHODE — Marge brute vs marge nette : la distinction fondamentale.**
>
> Dans ce projet on calcule la **marge brute** : prix de vente moins coût d'achat. C'est la seule marge que nos données permettent de calculer.
>
> La **marge nette** inclurait aussi les frais de livraison, les retours, les frais marketing, les salaires — ces données ne sont pas dans les CSV.
>
> **Impact sur l'interprétation :** un produit avec une marge brute de 60% peut être non rentable si les retours sont fréquents ou s'il nécessite des campagnes paid coûteuses. Mentionner cette limite lors de la restitution à M. Diallo.

### Question 2 — L'analyse RFM

| Dimension | Définition | Calcul dans ce projet |
|---|---|---|
| **R** — Recency | Depuis combien de jours la dernière commande | `aujourd'hui - max(order_date)` |
| **F** — Frequency | Nombre de commandes distinctes | `COUNT(DISTINCT order_id)` |
| **M** — Monetary | Montant total dépensé | `SUM(revenue)` |

> **MÉTHODE — RFM : segmentation comportementale, pas démographique.**
>
> Un client `segment_client = 'Premium'` peut avoir un R élevé (inactif depuis longtemps). RFM capture le **comportement récent**, pas le statut historique.
>
> Un client qui achetait beaucoup il y a 2 ans mais n'a pas commandé depuis 6 mois a un R mauvais malgré son historique — il est en train de churner. C'est ce type de signal que le `segment_client` existant ne détecte pas.

### Question 3 — Taux de conversion e-commerce

```
Taux de conversion = Nb sessions ayant abouti à une commande / Nb total de sessions × 100
```

Avec les données disponibles :
- `web_logs.csv` fournit les sessions visiteur
- `orders.csv` fournit les commandes
- On joint par `customer_id` pour identifier les sessions converties

> **MÉTHODE — Ne jamais analyser le taux de conversion global.**
>
> Un taux global de 2.5% ne dit rien d'actionnable. Il faut le décomposer par :
> - **Device** : mobile vs desktop (souvent 2x à 3x d'écart)
> - **Canal** : Paid Search convertit généralement mieux qu'Organic
> - **Catégorie** : les produits à fort prix convertissent moins
>
> La décomposition révèle les leviers. Le global les cache.

### Question 4 — Performance produit vs satisfaction client

| Analyse | Question clé | Données | Métrique principale |
|---|---|---|---|
| **Performance produit** | Ce produit est-il rentable et bien vendu ? | order_items + products | CA, marge, quantités |
| **Satisfaction client** | Ce produit plaît-il aux acheteurs ? | reviews | rating moyen, % avis < 3 |

> **MÉTHODE — Croiser les deux analyses révèle les vrais risques.**
>
> | CA | Satisfaction | Profil | Implication |
> |---|---|---|---|
> | Fort | Élevée | ⭐ Étoile | Protéger, augmenter la visibilité |
> | Fort | Basse | 💣 Bombe à retardement | Risque de retour massif et churn |
> | Faible | Élevée | 🚀 Potentiel sous-exploité | Booster la visibilité / promotion |
> | Faible | Basse | ❌ À déprioritiser | Analyser la cause racine |

### Question 5 — CA en hausse + avis en baisse : hypothèses à tester

**Hypothèse 1 — Nouveaux clients mal ciblés :**  
La croissance vient de canaux (Paid Ads) qui attirent des clients mal ciblés, qui commandent une fois, sont déçus et laissent de mauvais avis.  
→ *Tester : comparer la note moyenne des nouveaux clients vs clients fidèles.*

**Hypothèse 2 — Dégradation qualité produit :**  
Les bestsellers ont changé de fournisseur pour réduire les coûts → qualité perçue baisse.  
→ *Tester : comparer les ratings avant/après une date charnière sur les top produits.*

**Hypothèse 3 — Problèmes logistiques liés au volume :**  
Plus de commandes → délais de livraison plus longs → avis négatifs sur la livraison.  
→ *Tester : corrélation entre `delivery_delay` et `rating` dans reviews.*

**Hypothèse 4 — Canaux moins qualitatifs :**  
Des canaux moins ciblants (Social) ramenant des clients avec des attentes décalées.  
→ *Tester : comparer les ratings par canal d'acquisition.*

> **MÉTIER — Ces 4 hypothèses sont exactement ce qu'on va tester dans l'étape 3.** Les noter maintenant permet d'avoir un plan clair et de ne pas les oublier.

---
## 1. Mise en place de l'environnement

> **MÉTHODE — La palette de couleurs est définie une seule fois et réutilisée sur tous les graphiques.**
>
> Chaque couleur a un sens sémantique fixe :
> - Violet `#534AB7` : indicateur principal, neutre
> - Vert `#1D9E75` : résultat positif, objectif atteint
> - Orange `#EF9F27` : vigilance, zone d'attention
> - Rouge `#E24B4A` : alerte, problème à traiter
> - Gris `#888780` : référence, comparaison
>
> Quand M. Diallo voit du rouge, il sait que c'est une alerte — sans lire le titre du graphique.

In [ ]:


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings


import os
import sys

# Détecter si on est dans Colab ou en local
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = "/content/drive/MyDrive/DataProjectLab/projects/ecommerce_analytics/"
else:
    SAVE_PATH = "./outputs/"

# Chemin pour enregistrer les fichiers exportés
os.makedirs(SAVE_PATH, exist_ok=True)
print(f" Environnement Colab : {'Colab' if IN_COLAB else 'Local'}")
print(f" Dossier de travail : {SAVE_PATH}")

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.2f}".format)

COLORS = {
    "primary":   "#534AB7",
    "secondary": "#1D9E75",
    "warning":   "#EF9F27",
    "danger":    "#E24B4A",
    "neutral":   "#888780",
    "light":     "#EEEDFE",
}

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "#F9F9F8",
    "axes.grid":        True,
    "grid.alpha":       0.35,
    "font.size":        11,
})

print("Environnement prêt ✅")

---
## 1. Chargement des données

> **MÉTHODE — `parse_dates` est obligatoire sur toutes les colonnes temporelles.**
>
> Sans `parse_dates=['order_date']`, pandas charge la colonne comme une chaîne de caractères. Conséquences :
> - `orders['order_date'].dt.month` → `AttributeError`
> - `orders.sort_values('order_date')` → tri alphabétique (faux ordre chronologique)
> - Agrégation par mois impossible
>
> **Règle : `parse_dates` au `read_csv`, toujours, jamais après.**

In [ ]:
BASE_URL = "https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/main/projets/ecommerce_analytics/data/"

customers    = pd.read_csv(BASE_URL + "customers.csv",    parse_dates=["date_inscription"])
products     = pd.read_csv(BASE_URL + "products.csv")
orders       = pd.read_csv(BASE_URL + "orders.csv",       parse_dates=["order_date"])
order_items  = pd.read_csv(BASE_URL + "order_items.csv")
reviews      = pd.read_csv(BASE_URL + "reviews.csv")
web_logs     = pd.read_csv(BASE_URL + "web_logs.csv",     parse_dates=["timestamp"])
social_media = pd.read_csv(BASE_URL + "social_media.csv", parse_dates=["date"])

print(f"{'Table':<20} {'Lignes':>8} {'Colonnes':>10}")
print("-" * 42)
for nom, df in [("customers", customers), ("products", products), ("orders", orders),
                ("order_items", order_items), ("reviews", reviews),
                ("web_logs", web_logs), ("social_media", social_media)]:
    print(f"{nom:<20} {len(df):>8,} {len(df.columns):>10}")

> **INTERPRÉTATION — Que nous dit ce tableau ?**
>
> `orders` et `order_items` : l'écart (~2x) signifie que chaque commande contient en moyenne 2 produits.
>
> `web_logs` est la table la plus volumineuse. Un grand volume de sessions pour peu de commandes signifie un taux de conversion probablement faible — à mesurer dans l'étape 3.
>
> **Règle : le ratio entre les tailles des tables vous dit déjà quelque chose avant même d'analyser le contenu.**

---
## 2. Exploration rapide des tables

> **MÉTHODE — Explorer la structure avant d'analyser le contenu.**
>
> Pour chaque table, on vérifie 3 choses :
> 1. La granularité (1 ligne = quoi ?)
> 2. Les valeurs nulles (quelles colonnes ont des trous ?)
> 3. L'aperçu visuel (est-ce que les valeurs ont du sens ?)

In [ ]:
tables = {
    "customers":   customers,
    "products":    products,
    "orders":      orders,
    "order_items": order_items,
    "reviews":     reviews,
}

for nom, df in tables.items():
    print(f"\n{'='*55}")
    print(f"  TABLE : {nom.upper()}")
    print(f"{'='*55}")
    print(f"  Dimensions : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
    print(f"  Colonnes   : {df.columns.tolist()}")
    nulls = df.isnull().sum()
    if nulls.sum() > 0:
        print(f"  Nulls      : {nulls[nulls > 0].to_dict()}")
    else:
        print("  Nulls      : aucun")
    print("  Aperçu     :")
    display(df.head(3))

> **INTERPRÉTATION — Ce que l'exploration révèle**
>
> **Sur `products` :** `prix_unitaire` est le prix catalogue. Dans `order_items`, le `prix_unitaire` est le prix réel payé lors de la commande. Pour le CA réel, toujours utiliser `order_items.prix_unitaire * quantite`, jamais `products.prix_unitaire`.
>
> **Sur `orders` :** Le champ `order_status` est critique. Un statut mal encodé comme `'Livree '` (avec espace) ne matchera pas `'Livree'`. Ce sera vérifié et corrigé en étape 2.
>
> **Sur `reviews` :** La colonne `comment` est du texte libre. Elle ouvre la porte à une analyse de sentiment (NLP) avancée. Pour ce projet, on se concentre sur le `rating` numérique.

---
## 3. Granularité et intégrité référentielle

> **MÉTHODE — Vérifier la granularité de chaque table avant tout JOIN.**
>
> La granularité répond à la question : '1 ligne = quoi ?'
> Ne pas la connaître avant un JOIN peut créer des doublons silencieux qui faussent tous les calculs d'agrégation sans message d'erreur.

In [ ]:
print("GRANULARITÉ DES TABLES")
print("-" * 50)
print(f"  customers    : 1 ligne = 1 CLIENT UNIQUE")
print(f"    Nb clients   : {customers['customer_id'].nunique():,}")
print(f"    Doublons     : {len(customers) - customers['customer_id'].nunique():,}")
print()
print(f"  orders       : 1 ligne = 1 COMMANDE")
print(f"    Nb commandes : {orders['order_id'].nunique():,}")
print(f"    Période      : {orders['order_date'].min().date()} → {orders['order_date'].max().date()}")
print()
print(f"  order_items  : 1 ligne = 1 PRODUIT dans 1 COMMANDE")
print(f"    Nb lignes    : {len(order_items):,}")
print(f"    Nb produits/commande (moy) : {len(order_items)/order_items['order_id'].nunique():.1f}")
print()
print(f"  reviews      : 1 ligne = 1 AVIS CLIENT")
print(f"    Nb avis      : {len(reviews):,}")
print(f"    Note moyenne : {reviews['rating'].mean():.2f} / 5")
print(f"    Taux réponse : {len(reviews)/orders['order_id'].nunique()*100:.1f}% des commandes ont un avis")

In [ ]:
print("VÉRIFICATION DE L'INTÉGRITÉ RÉFÉRENTIELLE")
print("-" * 50)

orphans_clients  = orders[~orders["customer_id"].isin(customers["customer_id"])]
orphans_products = order_items[~order_items["product_id"].isin(products["product_id"])]
orphans_reviews  = reviews[~reviews["customer_id"].isin(orders["customer_id"])]

print(f"  Commandes avec customer_id inconnu : {len(orphans_clients)}")
print(f"  Lignes avec product_id inconnu     : {len(orphans_products)}")
print(f"  Avis avec customer_id inconnu      : {len(orphans_reviews)}")

total = len(orphans_clients) + len(orphans_products) + len(orphans_reviews)
print(f"\n  Intégrité référentielle : {'✅ OK' if total == 0 else f'⚠️ {total} anomalies détectées'}")

print(f"\n  Valeurs order_status : {sorted(orders['order_status'].unique().tolist())}")
print(f"  Valeurs canal        : {sorted(orders['canal'].unique().tolist())}")

> **MÉTHODE — La liste des valeurs de `order_status` et `canal`**
>
> On vérifie qu'il n'y a pas de valeurs parasites comme `'Livree '` (espace en fin) ou `'livree'` (minuscule). Ces variantes sont invisibles à l'œil mais cassent tous les filtres. Si détectées, elles seront normalisées avec `.str.strip().str.title()` en étape 2.

---
## 4. Construction du dataset consolidé et premiers KPIs

> **MÉTHODE — Partir de `order_items` comme table centrale.**
>
> On part de `order_items` car c'est la table à la granularité la plus fine (1 ligne = 1 produit dans 1 commande). En enrichissant avec `orders`, `products` et `customers`, on obtient toutes les informations nécessaires.
>
> **Pourquoi `inner join` entre order_items et orders ?**  
> Toute ligne order_items doit avoir une commande associée. Un `left join` génèrerait des NaN sur `order_date` — ce qui fausserait les analyses temporelles.
>
> **Pourquoi filtrer sur `order_status = 'Livree'` pour les KPIs financiers ?**  
> Annulées et Remboursées = aucun CA réel perçu. En cours = CA pas encore finalisé.  
> Inclure ces statuts surestimerait le CA et fausserait la marge.

In [ ]:
df = (
    order_items
    .merge(orders[["order_id", "customer_id", "order_date", "canal", "order_status", "montant_total"]],
           on="order_id", how="inner")
    .merge(products[["product_id", "nom_produit", "categorie", "cout_produit"]],
           on="product_id", how="left")
    .merge(customers[["customer_id", "pays", "segment_client"]],
           on="customer_id", how="left")
)

df["revenue"] = df["quantite"] * df["prix_unitaire"]
df["margin"]  = df["quantite"] * (df["prix_unitaire"] - df["cout_produit"])
df["mois"]    = df["order_date"].dt.to_period("M")

df_livrees = df[df["order_status"] == "Livree"].copy()

print(f"Dataset consolidé  : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
print(f"Commandes livrées  : {df_livrees['order_id'].nunique():,} commandes")
print(f"Toutes commandes   : {df['order_id'].nunique():,} commandes")
print(f"Poids des livrées  : {df_livrees['order_id'].nunique()/df['order_id'].nunique()*100:.1f}%")

### 4.1 — Tableau de bord global

In [ ]:
ca_total     = df_livrees["revenue"].sum()
marge_totale = df_livrees["margin"].sum()
nb_commandes = df_livrees["order_id"].nunique()
nb_clients   = df_livrees["customer_id"].nunique()
panier_moyen = df_livrees.groupby("order_id")["revenue"].sum().mean()
taux_marge   = marge_totale / ca_total * 100

print("=" * 55)
print("  TABLEAU DE BORD — ShopAfrica+ (commandes livrées)")
print("=" * 55)
print(f"  CA total              : {ca_total:>12,.0f} €")
print(f"  Marge brute totale    : {marge_totale:>12,.0f} €")
print(f"  Taux de marge moyen   : {taux_marge:>11.1f} %")
print(f"  Nb commandes livrées  : {nb_commandes:>12,}")
print(f"  Nb clients actifs     : {nb_clients:>12,}")
print(f"  Panier moyen          : {panier_moyen:>12.2f} €")
print("=" * 55)
print(f"\n  Période : {df_livrees['order_date'].min().date()} → {df_livrees['order_date'].max().date()}")

> **INTERPRÉTATION — Lecture du tableau de bord**
>
> **Le taux de marge brute (~37%) :** Pour chaque 100€ de ventes, 37€ restent après déduction du coût d'achat. En e-commerce, un taux de 30-45% est standard. C'est un signal sain — mais rappelons que c'est la marge **brute**, sans frais de livraison, marketing ni retours.
>
> **Nb clients actifs vs nb total clients :** La différence représente les clients inscrits mais n'ayant jamais acheté — le vivier de réactivation pour les campagnes email.

### 4.2 — Évolution mensuelle du CA

> **MÉTHODE — La ligne de tendance avec `np.polyfit`.**
>
> `np.polyfit(x, y, 1)` ajuste une droite de régression linéaire. Le coefficient `z[0]` donne la **pente** : la variation moyenne du CA par mois. Une pente négative = décroissance tendancielle malgré d'éventuels pics saisonniers.

In [ ]:
ca_mensuel = (
    df_livrees.groupby("mois")["revenue"]
    .sum().reset_index()
    .assign(mois_str=lambda x: x["mois"].astype(str))
)

idx = range(len(ca_mensuel))
z   = np.polyfit(idx, ca_mensuel["revenue"], 1)

fig, ax = plt.subplots(figsize=(13, 5))

ax.bar(idx, ca_mensuel["revenue"] / 1000,
       color=COLORS["primary"], alpha=0.75, edgecolor="white", width=0.8)

ax.plot(idx, np.poly1d(z)(idx) / 1000,
        color=COLORS["danger"], linewidth=2, linestyle="--",
        label=f"Tendance ({z[0]/1000:+.1f}k€/mois)")

ax.set_xticks(list(idx)[::3])
ax.set_xticklabels(ca_mensuel["mois_str"].iloc[::3], rotation=45, ha="right", fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.0f}k"))
ax.set_title("Évolution mensuelle du CA — ShopAfrica+ (commandes livrées)", fontweight="bold", fontsize=13)
ax.set_xlabel("Mois")
ax.set_ylabel("CA (k€)")
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

croissance = (ca_mensuel["revenue"].iloc[-1] - ca_mensuel["revenue"].iloc[0]) / ca_mensuel["revenue"].iloc[0] * 100
print(f"Croissance 1er → dernier mois  : {croissance:+.1f}%")
print(f"CA moyen mensuel               : {ca_mensuel['revenue'].mean()/1000:.0f}k€")
print(f"CA min (mois creux)            : {ca_mensuel['revenue'].min()/1000:.0f}k€")
print(f"CA max (mois pic)              : {ca_mensuel['revenue'].max()/1000:.0f}k€")
print(f"Coefficient de variation       : {ca_mensuel['revenue'].std()/ca_mensuel['revenue'].mean()*100:.1f}%")

> **INTERPRÉTATION — Évolution du CA mensuel**
>
> **La tendance (pente de la droite) :** C'est la première réponse à M. Diallo : *"Nos ventes augmentent, mais..."*. Si la pente est **négative**, le CA a une tendance structurelle à la baisse malgré des pics de saisonnalité — signal plus préoccupant qu'un seul mauvais mois.
>
> **Le coefficient de variation (CV) :** CV = écart-type / moyenne. Un CV > 30% signifie que la saisonnalité est marquée et que les moyennes annuelles sont peu significatives.
>
> **Ce qu'on dit à M. Diallo :** *"La tendance de fond est de X k€ par mois. La saisonnalité est forte avec un pic en décembre et un creux en été. Des campagnes proactives avant les creux seraient bénéfiques."*

### 4.3 — Top 5 catégories : CA vs Marge

> **MÉTHODE — Toujours combiner CA et marge dans la même visualisation.**
>
> Présenter uniquement le CA par catégorie est trompeur. Une catégorie peut dominer le CA mais avoir une marge faible. Le double graphique (CA gauche, taux de marge droit) répond aux deux questions en même temps : **'Qui vend le plus ?'** ET **'Qui gagne le plus ?'**

In [ ]:
top_cat = (
    df_livrees.groupby("categorie")
    .agg(ca=("revenue", "sum"), marge=("margin", "sum"), nb_commandes=("order_id", "nunique"))
    .assign(taux_marge=lambda x: x["marge"] / x["ca"] * 100)
    .sort_values("ca", ascending=False)
    .head(5).reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CA par catégorie
colors_cat = [COLORS["primary"] if i == 0 else "#534AB7aa" for i in range(len(top_cat))]
bars = axes[0].barh(top_cat["categorie"], top_cat["ca"] / 1000, color=colors_cat, alpha=0.85, edgecolor="white")
for bar, v in zip(bars, top_cat["ca"] / 1000):
    axes[0].text(v + 2, bar.get_y() + bar.get_height() / 2,
                 f"{v:.0f}k", va="center", fontsize=10, fontweight="bold", color=COLORS["primary"])
axes[0].set_title("Top 5 catégories — CA total", fontweight="bold")
axes[0].set_xlabel("CA (k€)")
axes[0].invert_yaxis()

# Taux de marge
colors_marge = [COLORS["secondary"] if v >= 35 else COLORS["warning"] if v >= 25 else COLORS["danger"] for v in top_cat["taux_marge"]]
axes[1].barh(top_cat["categorie"], top_cat["taux_marge"], color=colors_marge, alpha=0.85, edgecolor="white")
axes[1].axvline(top_cat["taux_marge"].mean(), color=COLORS["neutral"], linestyle="--",
                linewidth=1.5, label=f"Moy. {top_cat['taux_marge'].mean():.1f}%")
axes[1].set_title("Taux de marge par catégorie", fontweight="bold")
axes[1].set_xlabel("Taux de marge (%)")
axes[1].invert_yaxis()
axes[1].legend(fontsize=9)

plt.suptitle("Performance commerciale par catégorie", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

display(top_cat[["categorie", "ca", "marge", "taux_marge", "nb_commandes"]]
        .rename(columns={"ca": "CA (€)", "marge": "Marge (€)", "taux_marge": "Taux marge (%)"}))

> **INTERPRÉTATION — La matrice recommandée à M. Diallo**
>
> | CA | Marge | Profil | Action |
> |---|---|---|---|
> | Fort | Élevée | ⭐ Étoile | Protéger, augmenter la visibilité |
> | Fort | Basse | 💣 Bombe à retardement | Risque de retour massif et churn |
> | Faible | Élevée | 🚀 Potentiel sous-exploité | Booster la visibilité / promotion |
> | Faible | Basse | ❌ À déprioritiser | Analyser la cause racine |
>
> Cette matrice est plus puissante que n'importe quel classement par CA seul.

### 4.4 — Performance par canal d'acquisition

> **MÉTHODE — Le donut de répartition + le panier moyen : deux questions complémentaires.**
>
> Le donut répond à : *'Quel canal génère le plus de CA ?'*  
> Le panier moyen répond à : *'Quel canal génère les clients les plus dépensiers ?'*  
> Ces deux questions n'ont pas la même réponse.

In [ ]:
canal_perf = (
    df_livrees.groupby("canal")
    .agg(ca=("revenue", "sum"), nb_cmd=("order_id", "nunique"), panier_moy=("montant_total", "mean"))
    .assign(part_ca=lambda x: x["ca"] / x["ca"].sum() * 100)
    .sort_values("ca", ascending=False).reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors_canal = [COLORS["primary"], COLORS["secondary"], COLORS["warning"], COLORS["danger"], COLORS["neutral"]][:len(canal_perf)]

# Donut
axes[0].pie(canal_perf["ca"], labels=canal_perf["canal"], colors=colors_canal,
            autopct="%1.1f%%", wedgeprops=dict(edgecolor="white", linewidth=2),
            startangle=90, pctdistance=0.75, labeldistance=1.1)
axes[0].add_patch(plt.Circle((0, 0), 0.55, color="white"))
axes[0].set_title("Part du CA par canal", fontweight="bold")

# Panier moyen
axes[1].bar(canal_perf["canal"], canal_perf["panier_moy"], color=colors_canal, alpha=0.85, edgecolor="white")
axes[1].set_title("Panier moyen par canal (€)", fontweight="bold")
axes[1].set_ylabel("Panier moyen (€)")
axes[1].tick_params(axis="x", rotation=20)
for bar, v in zip(axes[1].patches, canal_perf["panier_moy"]):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                 f"{v:.0f}", ha="center", fontsize=9, fontweight="bold")

plt.suptitle("Performance commerciale par canal d'acquisition", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

display(canal_perf)

> **INTERPRÉTATION — Performance par canal**
>
> **Organic (SEO) :** C'est le canal le plus rentable sur le long terme — pas de coût variable par clic. Chaque euro investi en SEO continue de ramener des clients sans coût supplémentaire.
>
> **Paid Search :** Si son panier moyen est le plus élevé, ces clients sont mieux ciblés (intention d'achat forte). Même avec moins de volume, le ROI par euro dépensé peut être excellent.
>
> **Ce qu'on dit à M. Diallo :** *"Organic est votre moteur principal. Paid Search ramène des clients à plus fort panier moyen. Renforcer le SEO et optimiser les campagnes Paid Search vers les catégories à forte marge sont les deux leviers prioritaires."*

### 4.5 — Distribution des statuts de commande

> **MÉTHODE — Les statuts non-livrés sont un indicateur de qualité opérationnelle.**
>
> **Seuils d'alerte :**  
> Taux d'annulation > 10% → signal sur le processus de commande (stock, paiement, promesses non tenues).  
> Taux de remboursement > 5% → signal sur la qualité produit ou les descriptions (clients déçus à réception).

In [ ]:
statuts = orders["order_status"].value_counts()

fig, ax = plt.subplots(figsize=(8, 4))
colors_statuts = {"Livree": COLORS["secondary"], "En cours": COLORS["primary"],
                  "Annulee": COLORS["danger"], "Remboursee": COLORS["warning"]}
bars = ax.bar(statuts.index, statuts.values,
              color=[colors_statuts.get(s, COLORS["neutral"]) for s in statuts.index],
              alpha=0.85, edgecolor="white")
for bar, v in zip(bars, statuts.values):
    pct = v / statuts.sum() * 100
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10,
            f"{v:,}\n({pct:.1f}%)", ha="center", fontsize=9, fontweight="bold")
ax.set_title("Distribution des statuts de commande", fontweight="bold", fontsize=13)
ax.set_ylabel("Nombre de commandes")
ax.set_ylim(0, statuts.max() * 1.2)
plt.tight_layout()
plt.show()

taux_annul = statuts.get("Annulee", 0) / statuts.sum() * 100
taux_remb  = statuts.get("Remboursee", 0) / statuts.sum() * 100
print(f"Taux d'annulation  : {taux_annul:.1f}% {'⚠️ ALERTE' if taux_annul > 10 else '✅ OK'}")
print(f"Taux remboursement : {taux_remb:.1f}%  {'⚠️ ALERTE' if taux_remb > 5 else '✅ OK'}")
print(f"\nCA potentiellement perdu (annulations) : {orders[orders['order_status']=='Annulee']['montant_total'].sum():,.0f} €")

---
## 5. Synthèse business

> **MÉTHODE — La synthèse est le livrable le plus important du notebook.**
>
> M. Diallo ne lira pas le code. Il lira cette section.  
> Format obligatoire : **constats en chiffres + implications + actions recommandées.**

### Constat 1 — Tendance CA
**Chiffre :** La tendance linéaire sur la période est légèrement négative, masquée par une forte saisonnalité.  
**Implication :** Le CA stagne structurellement malgré des pics.  
**Action :** Activer des campagnes promotionnelles avant les creux saisonniers.

---

### Constat 2 — Divergence CA vs Marge par catégorie
**Chiffre :** Les catégories dominant le CA ont un taux de marge inférieur à la moyenne.  
**Implication :** La croissance se fait via les catégories les moins rentables.  
**Action :** Booster la visibilité des catégories à forte marge.

---

### Constat 3 — Organic est le moteur, Paid Search a le meilleur panier
**Chiffre :** Organic génère ~25% du CA. Paid Search a le panier moyen le plus élevé.  
**Implication :** Deux canaux complémentaires — volume vs valeur.  
**Action :** Investir dans le SEO + optimiser Paid Search sur les catégories à forte marge.

---

### Constat 4 — Qualité opérationnelle à surveiller
**Chiffre :** ~11% des commandes sont annulées ou remboursées.  
**Implication :** 1 commande sur 9 ne se concrétise pas — CA potentiellement perdu significatif.  
**Action :** Investiguer les causes (catégories, canaux) dans l'étape 3.

---

| Étape | Ce qui est préparé |
|---|---|
| Étape 2 | Nettoyage des données, dataset analytique propre |
| Étape 3 | Segmentation RFM, funnel de conversion, analyse satisfaction |
| Étape 4 | Dashboard Power BI |

---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.